# PyTorch 迁移学习实战

> 本 notebook 是 [Keras 迁移学习实战](./迁移学习.ipynb) 的 PyTorch 等价版本。

## 核心概念

迁移学习（Transfer Learning）将在大规模数据集上预训练的模型知识迁移到新任务，尤其适合数据量有限的场景。

## 主要策略

| 策略 | 说明 | 适用场景 |
|------|------|----------|
| 特征提取 | 冻结预训练层，只训练新分类头 | 数据量小，任务相似 |
| 微调 | 解冻部分/全部层，低学习率训练 | 数据量适中，需要适配 |
| 从头训练 | 只使用预训练架构 | 数据量大，任务差异大 |

In [ ]:
import copy
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader, TensorDataset

# 设置随机种子 / Set random seed
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 设备选择 / Device selection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch 版本: {torch.__version__}")
print(f"使用设备: {device}")

## 第一部分：训练基础模型（模型 A）

首先在 Fashion MNIST 上训练一个完整的分类模型，作为迁移学习的源模型。

In [ ]:
# 加载 Fashion MNIST 数据集 / Load Fashion MNIST dataset
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True,
    transform=torchvision.transforms.ToTensor()
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True,
    transform=torchvision.transforms.ToTensor()
)

# 划分训练集和验证集 / Split into training and validation sets
X_train_full = train_dataset.data.float() / 255.0  # (60000, 28, 28)
y_train_full = train_dataset.targets.long()
X_test = test_dataset.data.float() / 255.0
y_test = test_dataset.targets.long()

X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

# 类别名称 / Class names
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print(f"训练集: {X_train.shape}")
print(f"验证集: {X_valid.shape}")
print(f"测试集: {X_test.shape}")

In [ ]:
class FashionClassifier(nn.Module):
    """
    Fashion MNIST 分类模型 / Fashion MNIST classification model

    与 Keras Sequential 模型等价的 PyTorch 实现：
    Flatten(28*28) -> Dense(300, relu) -> Dense(100, relu) -> Dense(10, softmax)

    注意：PyTorch 的 CrossEntropyLoss 已内置 softmax，
    因此输出层不需要 softmax 激活 / Note: PyTorch CrossEntropyLoss includes
    softmax internally, so no softmax activation in the output layer.

    Parameters
    ----------
    output_dim : int
        输出类别数 / Number of output classes
    """
    def __init__(self, output_dim: int = 10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.dense_1 = nn.Linear(28 * 28, 300)
        self.dense_2 = nn.Linear(300, 100)
        self.output = nn.Linear(100, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        前向传播 / Forward pass

        Parameters
        ----------
        x : torch.Tensor
            输入张量，形状 (batch, 28, 28) / Input tensor of shape (batch, 28, 28)

        Returns
        -------
        torch.Tensor
            输出 logits，形状 (batch, output_dim) / Output logits of shape (batch, output_dim)
        """
        x = self.flatten(x)
        x = self.relu(self.dense_1(x))
        x = self.relu(self.dense_2(x))
        x = self.output(x)
        return x


def create_base_model(output_dim: int = 10) -> FashionClassifier:
    """
    创建基础分类模型 / Create the base classification model

    Returns
    -------
    FashionClassifier
        10 分类的 Fashion MNIST 模型 / 10-class Fashion MNIST model
    """
    model = FashionClassifier(output_dim=output_dim)
    return model


# 创建并查看基础模型 / Create and inspect the base model
model_A = create_base_model().to(device)
print(model_A)
print(f"\n总参数量: {sum(p.numel() for p in model_A.parameters()):,}")

In [ ]:
def train_model(
    model: nn.Module,
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_val: torch.Tensor,
    y_val: torch.Tensor,
    loss_fn: nn.Module,
    optimizer: optim.Optimizer,
    epochs: int = 20,
    batch_size: int = 32,
    patience: int = 5,
    verbose: bool = True
) -> Dict[str, List[float]]:
    """
    通用训练函数 / Generic training function

    等价于 Keras 的 model.fit()，包含早停机制 / Equivalent to Keras model.fit() with early stopping.

    Parameters
    ----------
    model : nn.Module
        PyTorch 模型 / PyTorch model
    X_train, y_train : torch.Tensor
        训练数据 / Training data
    X_val, y_val : torch.Tensor
        验证数据 / Validation data
    loss_fn : nn.Module
        损失函数 / Loss function
    optimizer : optim.Optimizer
        优化器 / Optimizer
    epochs : int
        最大训练轮数 / Maximum number of epochs
    batch_size : int
        批大小 / Batch size
    patience : int
        早停耐心值 / Early stopping patience
    verbose : bool
        是否打印训练过程 / Whether to print training progress

    Returns
    -------
    Dict[str, List[float]]
        包含训练和验证的损失与准确率历史 / Dictionary with training and validation loss/accuracy history
    """
    history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    best_val_loss = float('inf')
    best_state = None
    wait = 0

    # 创建 DataLoader / Create DataLoader
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        # 训练阶段 / Training phase
        model.train()
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * len(y_batch)

            # 计算准确率 / Compute accuracy
            with torch.no_grad():
                if y_pred.dim() == 2 and y_pred.shape[1] > 1:
                    # 多分类 / Multi-class
                    preds = y_pred.argmax(dim=1)
                else:
                    # 二分类 / Binary
                    preds = (torch.sigmoid(y_pred.squeeze()) >= 0.5).long()
                epoch_correct += (preds == y_batch).sum().item()
                epoch_total += len(y_batch)

        train_loss = epoch_loss / epoch_total
        train_acc = epoch_correct / epoch_total

        # 验证阶段 / Validation phase
        model.eval()
        with torch.no_grad():
            X_val_dev, y_val_dev = X_val.to(device), y_val.to(device)
            val_pred = model(X_val_dev)
            val_loss = loss_fn(val_pred, y_val_dev).item()

            if val_pred.dim() == 2 and val_pred.shape[1] > 1:
                val_preds = val_pred.argmax(dim=1)
            else:
                val_preds = (torch.sigmoid(val_pred.squeeze()) >= 0.5).long()
            val_acc = (val_preds == y_val_dev).float().mean().item()

        history['loss'].append(train_loss)
        history['accuracy'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        if verbose:
            print(f"Epoch {epoch+1:3d}/{epochs} - "
                  f"loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - "
                  f"val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

        # 早停 / Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print("\n早停触发！恢复最佳权重 / Early stopping triggered! Restoring best weights.")
                model.load_state_dict(best_state)
                break

    # 确保加载最佳权重 / Ensure best weights are loaded
    if best_state is not None:
        model.load_state_dict(best_state)

    return history

In [ ]:
def evaluate_model(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    loss_fn: nn.Module,
    batch_size: int = 256
) -> Tuple[float, float]:
    """
    评估模型 / Evaluate model

    等价于 Keras 的 model.evaluate() / Equivalent to Keras model.evaluate().

    Parameters
    ----------
    model : nn.Module
        PyTorch 模型 / PyTorch model
    X, y : torch.Tensor
        评估数据 / Evaluation data
    loss_fn : nn.Module
        损失函数 / Loss function
    batch_size : int
        批大小 / Batch size

    Returns
    -------
    Tuple[float, float]
        (损失, 准确率) / (loss, accuracy)
    """
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item() * len(y_batch)

            if y_pred.dim() == 2 and y_pred.shape[1] > 1:
                preds = y_pred.argmax(dim=1)
            else:
                preds = (torch.sigmoid(y_pred.squeeze()) >= 0.5).long()
            total_correct += (preds == y_batch).sum().item()
            total_samples += len(y_batch)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy

In [ ]:
# 训练基础模型 A / Train base model A
model_A = create_base_model().to(device)
loss_fn_A = nn.CrossEntropyLoss()  # 等价于 sparse_categorical_crossentropy
optimizer_A = optim.Adam(model_A.parameters(), lr=1e-3)

print("训练基础模型 A...")
history_A = train_model(
    model_A, X_train, y_train, X_valid, y_valid,
    loss_fn=loss_fn_A, optimizer=optimizer_A,
    epochs=20, batch_size=32, patience=5, verbose=True
)

# 评估 / Evaluate
test_loss_A, test_acc_A = evaluate_model(model_A, X_test, y_test, loss_fn_A)
print(f"\n模型 A 测试准确率: {test_acc_A:.4f}")

# 保存最佳模型权重 / Save best model weights
torch.save(model_A.state_dict(), 'model_A_best.pt')
print("模型 A 权重已保存至 model_A_best.pt")

In [ ]:
# 绘制训练曲线 / Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_A['accuracy'], label='训练')
axes[0].plot(history_A['val_accuracy'], label='验证')
axes[0].set_title('模型 A 准确率')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_A['loss'], label='训练')
axes[1].plot(history_A['val_loss'], label='验证')
axes[1].set_title('模型 A 损失')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 第二部分：迁移学习（模型 B）

创建新任务：二分类任务（区分上装类 vs 下装类）

- **上装类 (label=1)**: T-shirt, Pullover, Coat, Shirt
- **下装类 (label=0)**: Trouser, Dress, Sandal, Sneaker, Bag, Ankle boot

In [ ]:
# 创建二分类标签 / Create binary classification labels
# 上装类: T-shirt(0), Pullover(2), Coat(4), Shirt(6) -> 1
# 其他 -> 0
upper_class_indices = [0, 2, 4, 6]

def convert_to_binary(y: torch.Tensor) -> torch.Tensor:
    """
    将多分类标签转换为二分类标签 / Convert multi-class labels to binary labels

    Parameters
    ----------
    y : torch.Tensor
        原始多分类标签 / Original multi-class labels

    Returns
    -------
    torch.Tensor
        二分类标签 (0 或 1) / Binary labels (0 or 1)
    """
    mask = torch.zeros_like(y, dtype=torch.bool)
    for idx in upper_class_indices:
        mask |= (y == idx)
    return mask.float()

y_train_B = convert_to_binary(y_train)
y_valid_B = convert_to_binary(y_valid)
y_test_B = convert_to_binary(y_test)

print("二分类标签分布:")
print(f"训练集 - 上装: {y_train_B.sum():.0f}, 其他: {len(y_train_B) - y_train_B.sum():.0f}")
print(f"测试集 - 上装: {y_test_B.sum():.0f}, 其他: {len(y_test_B) - y_test_B.sum():.0f}")

### 2.1 特征提取：冻结预训练层

In [ ]:
# 加载预训练模型并深拷贝权重 / Load pretrained model and deep copy weights
# Keras: clone_model + set_weights  -->  PyTorch: copy.deepcopy(state_dict)
model_A_loaded = create_base_model().to(device)
model_A_loaded.load_state_dict(torch.load('model_A_best.pt', weights_only=True))

# 深拷贝权重，避免修改影响原始模型 / Deep copy weights to avoid modifying the original
saved_weights = copy.deepcopy(model_A_loaded.state_dict())

print("预训练模型加载完成")
print(f"层列表: {[name for name, _ in model_A_loaded.named_children()]}")

In [ ]:
# 创建迁移学习模型 B / Create transfer learning model B
# Keras: 移除最后输出层，添加新层  -->  PyTorch: 替换最后的 nn.Linear 层
model_B = FashionClassifier(output_dim=1).to(device)  # 二分类输出 / Binary output

# 复制除输出层外的预训练权重 / Copy pretrained weights except the output layer
pretrained_dict = saved_weights
model_dict = model_B.state_dict()

# 过滤掉输出层的权重 / Filter out output layer weights
transfer_dict = {
    k: v for k, v in pretrained_dict.items()
    if k in model_dict and 'output' not in k
}
model_dict.update(transfer_dict)
model_B.load_state_dict(model_dict)

print("模型 B 架构（替换了输出层）:")
print(model_B)
print(f"\n输出层: {model_B.output}")

In [ ]:
# 第一阶段：冻结预训练层，只训练新的输出层
# Phase 1: Freeze pretrained layers, only train the new output layer
#
# Keras:  layer.trainable = False
# PyTorch: param.requires_grad = False

def freeze_layers(model: nn.Module, freeze_until: Optional[str] = None) -> None:
    """
    冻结模型层 / Freeze model layers

    等价于 Keras 的 layer.trainable = False / Equivalent to Keras layer.trainable = False.

    Parameters
    ----------
    model : nn.Module
        PyTorch 模型 / PyTorch model
    freeze_until : str, optional
        冻结到该层为止（不含该层）/ Freeze layers up to (but not including) this layer name.
        如果为 None，冻结除最后输出层外的所有层 / If None, freeze all layers except the output.
    """
    if freeze_until is None:
        # 冻结除输出层外的所有层 / Freeze all layers except output
        for name, param in model.named_parameters():
            if 'output' not in name:
                param.requires_grad = False
    else:
        # 冻结到指定层 / Freeze up to specified layer
        freeze = True
        for name, param in model.named_parameters():
            if name == freeze_until:
                freeze = False
            param.requires_grad = not freeze


def unfreeze_layers(model: nn.Module) -> None:
    """
    解冻模型所有层 / Unfreeze all model layers

    等价于 Keras 的 layer.trainable = True / Equivalent to Keras layer.trainable = True.

    Parameters
    ----------
    model : nn.Module
        PyTorch 模型 / PyTorch model
    """
    for param in model.parameters():
        param.requires_grad = True


def print_trainable_status(model: nn.Module) -> None:
    """
    打印每层的训练状态 / Print trainable status of each layer
    """
    print("层的训练状态:")
    for name, param in model.named_parameters():
        print(f"  {name}: requires_grad={param.requires_grad}, shape={param.shape}")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"\n可训练参数: {trainable:,} / 总参数: {total:,}")


# 冻结预训练层 / Freeze pretrained layers
freeze_layers(model_B)
print_trainable_status(model_B)

In [ ]:
# 使用较小的学习率编译 / Compile with smaller learning rate
# 注意：只将 requires_grad=True 的参数传入优化器 / Note: Only pass requires_grad=True params to optimizer
loss_fn_B = nn.BCEWithLogitsLoss()  # 等价于 binary_crossentropy + sigmoid
optimizer_B = optim.SGD(
    filter(lambda p: p.requires_grad, model_B.parameters()),
    lr=1e-3
)

# 训练第一阶段（特征提取）/ Train phase 1 (feature extraction)
print("第一阶段：特征提取（冻结预训练层）...")
history_B_phase1 = train_model(
    model_B, X_train, y_train_B, X_valid, y_valid_B,
    loss_fn=loss_fn_B, optimizer=optimizer_B,
    epochs=10, batch_size=32, patience=10, verbose=True  # 不使用早停 / No early stopping in phase 1
)

# 评估 / Evaluate
phase1_loss, phase1_acc = evaluate_model(model_B, X_test, y_test_B, loss_fn_B)
print(f"\n第一阶段测试准确率: {phase1_acc:.4f}")

### 2.2 微调：解冻部分层

In [ ]:
# 第二阶段：解冻所有层进行微调 / Phase 2: Unfreeze all layers for fine-tuning
#
# Keras:  layer.trainable = True
# PyTorch: param.requires_grad = True

unfreeze_layers(model_B)
print_trainable_status(model_B)

# 使用更小的学习率防止破坏预训练权重 / Use smaller LR to avoid destroying pretrained weights
# Keras: 重新 compile 模型（自动重置优化器状态）
# PyTorch: 创建新的优化器（或使用参数组调整学习率）
optimizer_B_finetune = optim.SGD(model_B.parameters(), lr=1e-4)  # 更小的学习率

print("\n微调阶段学习率: 1e-4 (比特征提取阶段小 10 倍)")

In [ ]:
# 训练第二阶段（微调）/ Train phase 2 (fine-tuning)
print("第二阶段：微调全部层...")
history_B_phase2 = train_model(
    model_B, X_train, y_train_B, X_valid, y_valid_B,
    loss_fn=loss_fn_B, optimizer=optimizer_B_finetune,
    epochs=20, batch_size=32, patience=5, verbose=True
)

# 最终评估 / Final evaluation
final_loss, final_acc = evaluate_model(model_B, X_test, y_test_B, loss_fn_B)
print(f"\n最终测试准确率: {final_acc:.4f}")

## 第三部分：对比实验

In [ ]:
# 从头训练一个模型进行对比 / Train a model from scratch for comparison
model_scratch = FashionClassifier(output_dim=1).to(device)

loss_fn_scratch = nn.BCEWithLogitsLoss()
optimizer_scratch = optim.Adam(model_scratch.parameters(), lr=1e-3)

print("训练从头开始的模型...")
history_scratch = train_model(
    model_scratch, X_train, y_train_B, X_valid, y_valid_B,
    loss_fn=loss_fn_scratch, optimizer=optimizer_scratch,
    epochs=20, batch_size=32, patience=5, verbose=False
)

scratch_loss, scratch_acc = evaluate_model(model_scratch, X_test, y_test_B, loss_fn_scratch)
print(f"从头训练测试准确率: {scratch_acc:.4f}")

In [ ]:
# 可视化对比结果 / Visualize comparison results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 迁移学习两阶段 / Transfer learning two phases
all_acc = history_B_phase1['val_accuracy'] + history_B_phase2['val_accuracy']
all_loss = history_B_phase1['val_loss'] + history_B_phase2['val_loss']

axes[0].plot(all_acc, 'b-', label='迁移学习', linewidth=2)
axes[0].plot(history_scratch['val_accuracy'], 'r--', label='从头训练', linewidth=2)
axes[0].axvline(x=len(history_B_phase1['val_accuracy']), color='gray', linestyle=':', label='微调开始')
axes[0].set_title('验证准确率对比')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(all_loss, 'b-', label='迁移学习', linewidth=2)
axes[1].plot(history_scratch['val_loss'], 'r--', label='从头训练', linewidth=2)
axes[1].axvline(x=len(history_B_phase1['val_loss']), color='gray', linestyle=':', label='微调开始')
axes[1].set_title('验证损失对比')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('transfer_learning_comparison_pytorch.png', dpi=150, bbox_inches='tight')
plt.show()

# 结果汇总 / Results summary
print("\n" + "="*50)
print("结果汇总")
print("="*50)
print(f"迁移学习（微调后）: {final_acc:.4f}")
print(f"从头训练:          {scratch_acc:.4f}")
print(f"提升:              {(final_acc - scratch_acc)*100:+.2f}%")

## TF vs PyTorch 对照

### 核心概念映射

| 操作 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| 冻结层 | `layer.trainable = False` | `param.requires_grad = False` |
| 解冻层 | `layer.trainable = True` | `param.requires_grad = True` |
| 克隆模型 | `clone_model()` + `set_weights()` | `copy.deepcopy(state_dict)` |
| 替换输出层 | `model.layers.pop(); model.add(new_layer)` | `model.output = nn.Linear(...)` 或重新构建 |
| 多分类损失 | `sparse_categorical_crossentropy` | `nn.CrossEntropyLoss()` |
| 二分类损失 | `binary_crossentropy` | `nn.BCEWithLogitsLoss()` |
| 训练循环 | `model.fit()` | 手动循环 (for epoch in ...) |
| 评估 | `model.evaluate()` | 手动循环 + `model.eval()` |
| 早停 | `EarlyStopping` 回调 | 手动实现 patience 逻辑 |
| 模型保存 | `model.save()` / `.keras` | `torch.save(state_dict)` / `.pt` |
| 优化器重置 | 重新 `compile()` | 创建新优化器实例 |

### 关键差异

1. **训练循环**：Keras 封装了完整的训练循环（`fit`），PyTorch 需要手动编写，但灵活性更高
2. **损失函数**：PyTorch 的 `CrossEntropyLoss` 内置 softmax，`BCEWithLogitsLoss` 内置 sigmoid，因此模型输出层不需要激活函数
3. **冻结机制**：Keras 通过 `trainable` 属性控制，PyTorch 通过 `requires_grad` 控制；PyTorch 需要显式过滤参数传给优化器
4. **权重复制**：Keras 的 `clone_model` 只复制架构，需 `set_weights` 复制权重；PyTorch 的 `deepcopy(state_dict)` 一步完成
5. **优化器状态**：Keras 重新 `compile` 时自动重置优化器；PyTorch 需要创建新优化器实例来重置动量等状态

## 总结：迁移学习最佳实践

### 流程

1. **加载预训练模型**：使用 `copy.deepcopy(state_dict)` 保留权重
2. **替换输出层**：根据新任务修改 `nn.Linear` 层
3. **特征提取阶段**：`param.requires_grad = False` 冻结预训练层，训练新层
4. **微调阶段**：`param.requires_grad = True` 解冻部分/全部层，用小学习率训练

### 关键参数

| 参数 | 特征提取阶段 | 微调阶段 |
|------|-------------|----------|
| 学习率 | 正常 (1e-3) | 较小 (1e-4 ~ 1e-5) |
| 冻结层 | 全部预训练层 | 可选择性冻结底层 |
| Epoch | 较少 (5-10) | 可以更多 |

In [ ]:
# 验证代码正确性 / Verify code correctness
print("迁移学习模块测试完成")
print("\n关键要点:")
print("1. 迁移学习可以利用预训练模型的特征提取能力")
print("2. 分两阶段训练：先冻结后微调")
print("3. 微调时使用更小的学习率保护预训练权重")
print("4. 适合数据量有限但任务相似的场景")
print("5. PyTorch 中通过 requires_grad 控制冻结，需显式管理优化器参数")

## 练习

### 练习 1：选择性冻结

在微调阶段，尝试只解冻最后两层（`dense_2` 和 `output`），而保持 `dense_1` 冻结。比较以下三种策略的效果：

- 全部冻结（只训练输出层）
- 只解冻最后两层
- 全部解冻

提示：使用 `freeze_layers()` 函数的 `freeze_until` 参数，或手动设置 `requires_grad`。

```python
# 示例代码 / Example code
model_B_selective = FashionClassifier(output_dim=1).to(device)
# ... 加载预训练权重 ...

# 只解冻 dense_2 和 output / Only unfreeze dense_2 and output
for name, param in model_B_selective.named_parameters():
    if 'dense_1' in name or 'flatten' in name:
        param.requires_grad = False
    else:
        param.requires_grad = True
```

### 练习 2：不同学习率的影响

在微调阶段，分别使用学习率 `1e-2`、`1e-3`、`1e-4`、`1e-5` 进行训练，观察：

- 哪个学习率效果最好？
- 学习率过大时会发生什么？
- PyTorch 中如何使用 `optim.lr_scheduler` 动态调整学习率？

提示：尝试 `torch.optim.lr_scheduler.StepLR` 或 `CosineAnnealingLR`。

```python
# 示例代码 / Example code
from torch.optim.lr_scheduler import StepLR

optimizer = optim.SGD(model.parameters(), lr=1e-3)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

# 在每个 epoch 结束后调用 / Call after each epoch
# scheduler.step()
```

### 练习 3：小数据集场景

模拟数据量有限的场景：只使用 200 个训练样本，比较迁移学习与从头训练的差异。思考：

- 数据量减少时，迁移学习的优势是否更明显？
- 特征提取和微调哪个策略在小数据集上更稳定？

```python
# 示例代码 / Example code
small_X_train = X_train[:200]
small_y_train_B = y_train_B[:200]

# 分别训练迁移学习模型和从头训练模型，比较结果
```